# CS781 Phase 6 — Notebook P6_01: Summarization data, statistics, chunk alignment, baselines

No neural network is trained here. The notebook prepares the data once for P6_02:

| Step | What happens |
|---|---|
| 1 | download `Phase_6_Summ` from Hugging Face, parse id / document / summary (test has no summaries) |
| 2 | checks: empty / duplicate ids, duplicate texts, train↔test overlap |
| 3 | statistics: document and summary lengths, compression ratio, a length model `summary_words ≈ a·doc_words^b` |
| 4 | hold out a **validation split** (5% of training documents; no official dev set in this phase) |
| 5 | **chunk alignment**: every document is cut into chunks of ≤ 600 tokens on sentence boundaries; every summary sentence is assigned to the chunk that best contains its content words (IDF-weighted recall) → (chunk → chunk-summary) training pairs that fit our 1024-token models |
| 6 | where in the document does the summary come from (by chunk position)? |
| 7 | baselines on validation with ROUGE-1 / ROUGE-L: lead-N words, lead spread over chunks, alignment oracle |
| 8 | save token files for P6_02 |

**Kaggle settings**: Accelerator **None**, Internet **ON**, secret `HF`. **Input**: the **NB1 output** (tokenizer).
**Output**: `/kaggle/working/cs781_p1/p6/data/…` → attach to P6_02.

In [1]:
!pip -q install tokenizers

In [2]:
# ============================================================================================
# SHARED SETUP  —  paste this SAME cell at the top of every Phase-1 notebook.
# Works on Kaggle (Save & Commit or interactive), Colab (Drive) and local machines.
# ============================================================================================
import os, sys, json, time, glob, shutil, random, math, re, zlib, hashlib, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np

os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

SESSION_START = time.time()
TIME_BUDGET_H = 11.0   # Kaggle kills a session at 12h and then saves NOTHING -> every long loop stops itself before this

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
PROJECT   = "cs781_p1"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT, SCRATCH = Path("/content/drive/MyDrive") / PROJECT, Path("/content/scratch")   # Drive = persistent
elif IN_KAGGLE:
    ROOT, SCRATCH = Path("/kaggle/working") / PROJECT, Path("/tmp/cs781_scratch")       # working = saved on commit
else:
    ROOT, SCRATCH = Path("./") / PROJECT, Path("./scratch")
ROOT    = Path(os.environ.get("CS781_ROOT", ROOT))
SCRATCH = Path(os.environ.get("CS781_SCRATCH", SCRATCH))
ROOT.mkdir(parents=True, exist_ok=True); SCRATCH.mkdir(parents=True, exist_ok=True)

ENV = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"env={ENV}  ROOT={ROOT}  SCRATCH={SCRATCH}")
for p in {ROOT, SCRATCH}:
    du = shutil.disk_usage(p); print(f"  disk @ {p}: free {du.free/1e9:.1f} GB / total {du.total/1e9:.1f} GB")


# ---------------------------------------------------------------- small utilities
def time_left_h():
    return TIME_BUDGET_H - (time.time() - SESSION_START) / 3600

def save_json(obj, path):
    """Atomic write: a crash mid-write can never leave a half-written (corrupt) state file."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists(): return default
    with open(path, encoding="utf-8") as f: return json.load(f)

def restore_from_kaggle_inputs(subdirs):
    """Kaggle 'Save & Commit' starts every run with an EMPTY /kaggle/working.
    To resume, attach the output of the previous version (Add Input -> Your Work -> Notebook output).
    This copies <PROJECT>/<subdir> from any attached input into ROOT (never overwriting newer local files)."""
    if not IN_KAGGLE: return
    for sub in subdirs:
        for cand in sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{sub}", recursive=True)):
            n = 0
            for src in Path(cand).rglob("*"):
                if src.is_file():
                    dst = ROOT / sub / src.relative_to(cand)
                    if not dst.exists():
                        dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(src, dst); n += 1
            print(f"restored {n} files: {cand} -> {ROOT/sub}")

def find_artifact(rel):
    """Look for ROOT/rel first, then inside any attached Kaggle input."""
    p = ROOT / rel
    if p.exists(): return p
    if IN_KAGGLE:
        hits = sorted(glob.glob(f"/kaggle/input/**/{PROJECT}/{rel}", recursive=True))
        if hits: return Path(hits[0])
    return None

def get_hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF")
        if IN_COLAB:
            from google.colab import userdata
            return userdata.get("HF")
    except Exception as e:
        print("Could not read secret 'HF':", e)
    return None

HF_TOKEN = get_hf_token()
if HF_TOKEN: os.environ["HF_TOKEN"] = HF_TOKEN
print("HF token found:", bool(HF_TOKEN))

# Optional: a PRIVATE HF dataset repo used only to move YOUR OWN artifacts between accounts/sessions.
HF_ARTIFACT_REPO = None   # e.g. "your-hf-username/cs781-p1-artifacts"

def hf_push(subdir):
    if not HF_ARTIFACT_REPO: print("HF_ARTIFACT_REPO not set; skip push"); return
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_ARTIFACT_REPO, repo_type="dataset", private=True, exist_ok=True)
    api.upload_folder(folder_path=str(ROOT / subdir), path_in_repo=subdir,
                      repo_id=HF_ARTIFACT_REPO, repo_type="dataset")
    print("pushed", subdir)

def hf_pull(subdir):
    if not HF_ARTIFACT_REPO: return
    from huggingface_hub import snapshot_download
    snapshot_download(HF_ARTIFACT_REPO, repo_type="dataset", allow_patterns=[f"{subdir}/*", f"{subdir}/**"],
                      local_dir=str(ROOT), token=HF_TOKEN)
    print("pulled", subdir)


# ---------------------------------------------------------------- data utilities
# These MUST be identical in every notebook (the train/val split and cleaning depend on them).
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
SOURCE_NAMES = ["acts", "high_courts", "supreme_court", "tribunals"]   # processing order (matters for dedup)
VAL_PER_10K  = 100            # 1% of documents of every source are held out (document-level split)
PROMPT_WORDS, CONT_WORDS = 512, 128

def extract_text(rec):
    if rec is None: return ""
    if isinstance(rec, str): return rec
    if isinstance(rec, list): return "\n".join(extract_text(r) for r in rec)
    if isinstance(rec, dict):
        for k in ("text", "content", "document", "judgment", "judgement", "body"):
            if k in rec: return extract_text(rec[k])
    return ""

def iter_docs(path):
    """Stream documents from a (possibly multi-GB) JSON file WITHOUT loading it all into RAM.
    Handles: [ {...}, ... ]   |   { id: {...}, ... }   |   JSON-lines."""
    import ijson
    with open(path, "rb") as f:
        head = f.read(1 << 16).lstrip()
        f.seek(0)
        if head[:1] == b"[":
            for rec in ijson.items(f, "item", use_float=True):
                yield extract_text(rec)
            return
        first_line = f.readline(50_000_000); second = f.read(1 << 10).lstrip()[:1]
        f.seek(0)
        is_jsonl = False
        if first_line.endswith(b"\n") and second == b"{":
            try: json.loads(first_line); is_jsonl = True
            except Exception: pass
        if is_jsonl:
            for line in f:
                if line.strip(): yield extract_text(json.loads(line))
        else:
            for _, rec in ijson.kvitems(f, "", use_float=True):
                yield extract_text(rec)

_ZW = re.compile(r"[​‌‍﻿­]")
_WS_ALL = re.compile(r"\s+")
_HSPACE = re.compile(r"[^\S\n]+")
_MANY_NL = re.compile(r"\n\s*\n\s*\n+")

def normalize(text, mode):
    """'collapse_all'  -> every whitespace run becomes one space (text looks like ' '.join(text.split()))
       'keep_newlines' -> collapse spaces/tabs, keep at most one blank line"""
    text = _ZW.sub("", text)
    if mode == "collapse_all":
        return _WS_ALL.sub(" ", text).strip()
    text = _HSPACE.sub(" ", text.replace("\r\n", "\n").replace("\r", "\n"))
    text = "\n".join(l.strip() for l in text.split("\n"))
    return _MANY_NL.sub("\n\n", text).strip()

def hash64(text):
    return int.from_bytes(hashlib.blake2b(text.encode("utf-8", "ignore"), digest_size=8).digest(), "little")

def doc_split(source, idx):
    return "val" if zlib.crc32(f"{source}:{idx}".encode()) % 10000 < VAL_PER_10K else "train"

def list_phase1_files(local_raw_dir=None):
    """Return {'train': {source: repo_path}, 'test': repo_path, 'sizes': {repo_path: bytes}}."""
    if local_raw_dir:
        files = [str(Path(p).relative_to(local_raw_dir)) for p in glob.glob(f"{local_raw_dir}/**/*.json*", recursive=True)]
        sizes = {f: os.path.getsize(Path(local_raw_dir) / f) for f in files}
    else:
        from huggingface_hub import HfApi
        api = HfApi(token=HF_TOKEN)
        try:
            entries = list(api.list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                              path_in_repo="Phase_1_Pretraining", recursive=True))
            files = [e.path for e in entries if getattr(e, "size", None) is not None]
            sizes = {e.path: e.size for e in entries if getattr(e, "size", None) is not None}
        except Exception as e:
            print("list_repo_tree failed (", e, ") -> falling back to list_repo_files")
            files = [f for f in api.list_repo_files(HF_DATASET_REPO, repo_type="dataset") if f.startswith("Phase_1")]
            sizes = {}
    out = {"train": {}, "test": None, "sizes": sizes}
    for f in files:
        name = Path(f).name.lower()
        if not name.endswith((".json", ".jsonl")): continue
        if "test" in name: out["test"] = f
        for s in SOURCE_NAMES:
            if name.startswith("train") and s in name: out["train"][s] = f
    return out

def get_raw_file(repo_path, local_raw_dir=None):
    if local_raw_dir:
        return Path(local_raw_dir) / repo_path
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, repo_path, repo_type="dataset", token=HF_TOKEN,
                                local_dir=str(SCRATCH / "raw")))

def drop_raw_file(path, local_raw_dir=None):
    if local_raw_dir: return          # never delete user-provided files
    try: os.remove(path); print("deleted raw", path)
    except OSError: pass

env=kaggle  ROOT=/kaggle/working/cs781_p1  SCRATCH=/tmp/cs781_scratch
  disk @ /kaggle/working/cs781_p1: free 20.9 GB / total 21.0 GB
  disk @ /tmp/cs781_scratch: free 1126.1 GB / total 8758.4 GB
HF token found: True


## 0. Configuration and download

In [3]:
import pandas as pd
try: display
except NameError: display = print

P6_DATA = ROOT / "p6" / "data"; P6_DATA.mkdir(parents=True, exist_ok=True)
P6_REPO_DIR = "Phase_6_Summ"
P6_FILES = {"train": "Train/train.json", "test": "Test/test.json"}
TOKENIZER = "bpe32768"
SRC_TOK = 600          # max tokens of a document chunk (model input)
TGT_TOK = 384          # max tokens of a chunk summary (model output); 600 + 384 + separators < 1024 = Phase-1 context
VAL_FRAC = 0.05
P6_LOCAL = os.environ.get("CS781_P6_LOCAL")

def get_p6_file(rel):
    if P6_LOCAL: return Path(P6_LOCAL) / rel
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(HF_DATASET_REPO, f"{P6_REPO_DIR}/{rel}", repo_type="dataset", token=HF_TOKEN, local_dir=str(SCRATCH / "p6raw")))

PATHS = {k: get_p6_file(v) for k, v in P6_FILES.items()}
for k, p in PATHS.items(): print(f"{k}: {p}  ({os.path.getsize(p)/1e6:.1f} MB)")

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

train: /tmp/cs781_scratch/p6raw/Phase_6_Summ/Train/train.json  (210.6 MB)
test: /tmp/cs781_scratch/p6raw/Phase_6_Summ/Test/test.json  (2.7 MB)


## 1. Parse (flexible field names; test summaries are never read)

In [4]:
ID_KEYS, DOC_KEYS, SUM_KEYS = ("id", "doc_id", "case_id", "name"), ("document", "text", "judgment", "doc", "source"), ("summary", "abstract", "target", "reference")
def first_key(r, keys): return next((k for k in keys if k in r), None)
def as_text(x): return " ".join(map(str, x)) if isinstance(x, list) else ("" if x is None else str(x))

DATA = {}
for split, p in PATHS.items():
    with open(p, encoding="utf-8") as f: obj = json.load(f)
    recs = list(obj.items()) if isinstance(obj, dict) else [(None, r) for r in obj]
    sample = recs[0][1]
    print(f"{split}: {len(recs)} records; keys {list(sample)[:10] if isinstance(sample, dict) else type(sample).__name__}")
    ids, docs, sums = [], [], []
    for k, r in recs:
        if not isinstance(r, dict): r = {"document": r}
        ik, dk = first_key(r, ID_KEYS), first_key(r, DOC_KEYS)
        ids.append(str(r[ik]) if ik else str(k)); docs.append(normalize(as_text(r.get(dk)), "collapse_all"))
        if split != "test": sums.append(normalize(as_text(r.get(first_key(r, SUM_KEYS))), "collapse_all"))
    DATA[split] = {"ids": ids, "docs": docs, "sums": sums if split != "test" else None}
    del obj; gc.collect()
    print(f"   first document: {docs[0][:300]!r}")
    if split != "test": print(f"   first summary:  {sums[0][:300]!r}")

train: 7030 records; keys ['id', 'document', 'summary']
   first document: 'IVIL Appeal No.768 (NT) of 1977 etc. From the judgment Order dated 9.10.1975 of the Madhya Pradesh High Court in M.C.C. No.144 of 1966. V.A. Bobde, B.R.Aggarwala and U.A.Rana for the Appellant. S.V.Deshpande and S.K.Agnihotri for the Respondents. The Judgment of the Court was delivered by RANGANATH '
   first summary:  'The appellant, a manufacturer of cement, entered into an agreement with the Cement Manufacturing Company of India Limited, for sale of cement. Under the agreement, the appellant was to sell its cement only through the marketing company, and certain sums would be paid for the cement supplied by the m'
test: 100 records; keys ['id', 'document']
   first document: 'Tax Reference Cases Nos. 3 to 5 of 1975 Tax Reference under Section 257 of the Income tax Act, 1961 made by the Income tax Appellate Tribunal, Cuttack Bench, Cuttack P.A. Francis, Champat Rai and Miss A. Subhashini for the Appellant. C.S

## 2. Checks

In [5]:
for s, d in DATA.items():
    n_rep = len(d["ids"]) - len(set(d["ids"]))
    print(f"{s}: {len(d['ids'])} docs | repeated ids {n_rep} | empty documents {sum(not x for x in d['docs'])}"
          + (f" | empty summaries {sum(not x for x in d['sums'])}" if d["sums"] else ""))
assert len(set(DATA["test"]["ids"])) == len(DATA["test"]["ids"]), "test ids must be unique"
tr = DATA["train"]; keep, seen = [], set()
for i, (did, doc, sm) in enumerate(zip(tr["ids"], tr["docs"], tr["sums"])):
    key = hash64(doc[:20000])
    if not doc or not sm or did in seen or key in seen: continue
    seen.add(did); seen.add(key); keep.append(i)
print(f"train: kept {len(keep)} of {len(tr['ids'])} (dropped empty / repeated id / duplicate text)")
for k in ("ids", "docs", "sums"): tr[k] = [tr[k][i] for i in keep]
test_h = {hash64(x[:20000]) for x in DATA["test"]["docs"]}
OVERLAP_TEST_IN_TRAIN = sum(hash64(x[:20000]) in test_h for x in tr["docs"])
print("training documents identical to a test document:", OVERLAP_TEST_IN_TRAIN, "(removed)" if OVERLAP_TEST_IN_TRAIN else "")
if OVERLAP_TEST_IN_TRAIN:
    keep = [i for i, x in enumerate(tr["docs"]) if hash64(x[:20000]) not in test_h]
    for k in ("ids", "docs", "sums"): tr[k] = [tr[k][i] for i in keep]

train: 7030 docs | repeated ids 0 | empty documents 0 | empty summaries 2
test: 100 docs | repeated ids 0 | empty documents 0
train: kept 7001 of 7030 (dropped empty / repeated id / duplicate text)
training documents identical to a test document: 0 


## 3. Lengths and compression
A length model `summary_words ≈ a · doc_words^b` (fit in log space) lets P6_02 decide how long each generated summary
should be: ROUGE F1 rewards matching the reference length; BERTScore (RoBERTa, max 512 tokens) only sees the first
≈ 400 words of each summary, so the **beginning** of the summary matters most for the primary metric.

In [6]:
dw = np.array([len(x.split()) for x in tr["docs"]]); sw = np.array([len(x.split()) for x in tr["sums"]])
tw = np.array([len(x.split()) for x in DATA["test"]["docs"]])
rows = [{"split": "train documents", **pd.Series(dw).describe(percentiles=[.1, .5, .9]).to_dict()},
        {"split": "train summaries", **pd.Series(sw).describe(percentiles=[.1, .5, .9]).to_dict()},
        {"split": "test documents", **pd.Series(tw).describe(percentiles=[.1, .5, .9]).to_dict()}]
display(pd.DataFrame(rows).set_index("split").round(1))
ratio = sw / np.maximum(dw, 1)
print(f"summary / document word ratio: median {np.median(ratio):.3f}, p10 {np.percentile(ratio, 10):.3f}, p90 {np.percentile(ratio, 90):.3f}")
b, loga = np.polyfit(np.log(np.maximum(dw, 10)), np.log(np.maximum(sw, 5)), 1)
LEN_MODEL = {"a": float(np.exp(loga)), "b": float(b)}
pred = LEN_MODEL["a"] * np.maximum(dw, 10) ** LEN_MODEL["b"]
print(f"length model: summary_words ≈ {LEN_MODEL['a']:.2f} · doc_words^{LEN_MODEL['b']:.3f} | median abs error {np.median(np.abs(pred - sw)):.0f} words "
      f"(vs {np.median(np.abs(np.median(sw) - sw)):.0f} for a constant)")
def is_lower(t): s = t[:5000]; return sum(c.isupper() for c in s) < 0.002 * max(1, len(s))
print("lower-case share — documents:", round(float(np.mean([is_lower(x) for x in tr['docs']])), 3),
      "| summaries:", round(float(np.mean([is_lower(x) for x in tr['sums']])), 3), "| test:", round(float(np.mean([is_lower(x) for x in DATA['test']['docs']])), 3))
print("\nhow summaries begin (first 25 words of 5 summaries):")
for x in tr["sums"][:5]: print("  •", " ".join(x.split()[:25]))

,count,mean,std,min,10%,50%,90%,max
split,,,,,,,,
train documents,7001.0,4369.5,5116.4,93.0,1348.0,3131.0,7851.0,139645.0
train summaries,7001.0,841.1,900.6,26.0,293.0,639.0,1501.0,27768.0
test documents,100.0,4782.7,5443.9,511.0,1503.6,3171.0,8562.8,39695.0


summary / document word ratio: median 0.218, p10 0.093, p90 0.396
length model: summary_words ≈ 4.62 · doc_words^0.612 | median abs error 209 words (vs 251 for a constant)
lower-case share — documents: 0.0 | summaries: 0.0 | test: 0.0

how summaries begin (first 25 words of 5 summaries):
  • The appellant, a manufacturer of cement, entered into an agreement with the Cement Manufacturing Company of India Limited, for sale of cement. Under the agreement,
  • The appellant executed a usufructuary mortgage of his house and continued to reside in it as a tenant under a lease obtained from the mortgagee.
  • The appellant had filed a suit in the High Court of Calcutta for a declaration that the properties set out in the schedule belonged to
  • The appellant was injured in a road accident on 22.1.1989, and a claim petition was filed belatedly on 15.3.1990 with a prayer for condonation of
  • Some persons were plying their business by squatting on pavement in front of a hospital in Delhi and 

## 4. Validation split (5% of training documents, fixed by a hash of the id)

In [7]:
is_val = np.array([zlib.crc32(i.encode()) % 10000 < VAL_FRAC * 10000 for i in tr["ids"]])
DATA["val"] = {k: [v for v, f in zip(tr[k], is_val) if f] for k in ("ids", "docs", "sums")}
DATA["train"] = {k: [v for v, f in zip(tr[k], is_val) if not f] for k in ("ids", "docs", "sums")}
print({s: len(DATA[s]["ids"]) for s in ("train", "val", "test")})

{'train': 6646, 'val': 355, 'test': 100}


## 5. Chunk alignment
* **Chunks**: the document is split into sentences, sentences are packed into chunks of ≤ `SRC_TOK` tokens (a sentence
  longer than that is cut).
* **Alignment**: each summary sentence goes to the chunk containing the largest IDF-weighted share of its content
  words; a chunk's target = its sentences in summary order (≤ `TGT_TOK` tokens).
* Result: (chunk → chunk summary) pairs. At test time P6_02 summarises every chunk and joins the pieces in order —
  "chunk-level summaries followed by document-level assembly".

In [8]:
from tokenizers import Tokenizer
TOK_PATH = find_artifact(f"tokenizer/{TOKENIZER}/tokenizer.json")
assert TOK_PATH is not None, "attach the NB1 output (tokenizer/bpe32768/tokenizer.json)"
TOK = Tokenizer.from_file(str(TOK_PATH)); print("tokenizer:", TOK_PATH, "| vocab", TOK.get_vocab_size())

SENT_RE = re.compile(r"(?<=[.?!;])\s+(?=[A-Z0-9(\"'\[])")
def sentences(t): return [s for s in SENT_RE.split(t) if s.strip()]
STOP = set("the of and to in a is that be by for this on with as it was an are or which not at from he his has have had been were any its their they there such shall may would under said also into than other these those upon but so no if all".split())
WORD_RE = re.compile(r"[a-z0-9]+")
def content(t): return {w for w in WORD_RE.findall(t.lower()) if len(w) > 2 and w not in STOP}

df_cnt = Counter()
for x in DATA["train"]["docs"][:5000]: df_cnt.update(content(x[:30000]))
N_DF = min(5000, len(DATA["train"]["docs"]))
def idf(w): return math.log((N_DF + 1) / (df_cnt.get(w, 0) + 1)) + 1.0

def chunk_document(doc):
    sents = sentences(doc) or [doc]
    lens = [len(e.ids) for e in TOK.encode_batch(sents)]
    chunks, cur, cur_len = [], [], 0
    for s, L in zip(sents, lens):
        if L > SRC_TOK:                                      # very long "sentence": cut by words
            if cur: chunks.append(" ".join(cur)); cur, cur_len = [], 0
            w = s.split(); step = max(1, int(len(w) * SRC_TOK / L) - 5)
            chunks += [" ".join(w[i:i + step]) for i in range(0, len(w), step)]
            continue
        if cur_len + L > SRC_TOK and cur:
            chunks.append(" ".join(cur)); cur, cur_len = [], 0
        cur.append(s); cur_len += L
    if cur: chunks.append(" ".join(cur))
    return chunks

def bigrams(t):
    w = WORD_RE.findall(t.lower()); return set(zip(w, w[1:]))

def align(chunks, summary):
    """score(sentence, chunk) = 0.7 · IDF-weighted share of its content words in the chunk + 0.3 · share of its word bigrams"""
    cw = [content(c) for c in chunks]; cb = [bigrams(c) for c in chunks]; out = [[] for _ in chunks]; scores = []; j = 0
    for s in sentences(summary):
        ws, bs = content(s), bigrams(s)
        if not ws:                                           # no content words: stays with the previous sentence
            out[j].append(s); continue
        tot = sum(idf(w) for w in ws)
        sc = [0.7 * sum(idf(w) for w in ws & c) / tot + 0.3 * (len(bs & b2) / len(bs) if bs else 0) for c, b2 in zip(cw, cb)]
        j = int(np.argmax(sc)); out[j].append(s); scores.append(sc[j])
    return [" ".join(x) for x in out], scores

def build_pairs(split):
    d = DATA[split]; src, tgt, doc_idx, chk_idx, n_chunks, sc_all, pos_all = [], [], [], [], [], [], []
    for i, doc in enumerate(d["docs"]):
        ch = chunk_document(doc)
        tg, sc = align(ch, d["sums"][i]) if d["sums"] is not None else ([""] * len(ch), [])
        sc_all += sc
        if d["sums"] is not None:
            for j, t in enumerate(tg): pos_all += [j / max(1, len(ch) - 1)] * len(sentences(t)) if len(ch) > 1 else [0.0] * len(sentences(t))
        for j, (c, t) in enumerate(zip(ch, tg)):
            src.append(c); tgt.append(t); doc_idx.append(i); chk_idx.append(j); n_chunks.append(len(ch))
    return src, tgt, np.array(doc_idx), np.array(chk_idx), np.array(n_chunks), np.array(sc_all), np.array(pos_all)

PAIRS = {}
for s in ("train", "val", "test"):
    t0 = time.time(); PAIRS[s] = build_pairs(s)
    src, tgt, di, ci, nc, sc, pos = PAIRS[s]
    msg = f"{s}: {len(src)} chunks from {len(DATA[s]['ids'])} docs (chunks/doc mean {len(src)/max(1,len(DATA[s]['ids'])):.1f}, max {nc.max()})"
    if DATA[s]["sums"] is not None:
        empty = np.mean([not t for t in tgt])
        msg += f" | chunks with an empty target {empty:.1%} | summary-sentence alignment score median {np.median(sc):.2f}, < 0.3: {np.mean(sc < 0.3):.1%}"
    print(msg + f" | {time.time()-t0:.0f}s")

tokenizer: /kaggle/input/datasets/aaravoswal/cs781-p1-nb1/cs781_p1/tokenizer/bpe32768/tokenizer.json | vocab 32768
train: 62491 chunks from 6646 docs (chunks/doc mean 9.4, max 285) | chunks with an empty target 23.9% | summary-sentence alignment score median 0.83, < 0.3: 6.2% | 106s
val: 3300 chunks from 355 docs (chunks/doc mean 9.3, max 98) | chunks with an empty target 23.3% | summary-sentence alignment score median 0.83, < 0.3: 5.6% | 6s
test: 1020 chunks from 100 docs (chunks/doc mean 10.2, max 81) | 1s


## 6. Where in the document does the summary come from?
Share of summary sentences aligned to each tenth of the document (by chunk position).

In [9]:
pos = PAIRS["train"][6]
hist = np.histogram(pos, bins=10, range=(0, 1))[0]
print("share of summary sentences by relative document position (0 = start, 1 = end):",
      [round(float(x), 3) for x in hist / max(1, hist.sum())])

share of summary sentences by relative document position (0 = start, 1 = end): [0.22, 0.066, 0.084, 0.066, 0.056, 0.122, 0.083, 0.075, 0.099, 0.128]


## 7. Tokenise pairs and save
Pair files per split: `{split}_src_tokens.bin` / `_offsets.npy`, `{split}_tgt_tokens.bin` / `_offsets.npy` (uint16),
`{split}_pairs.npz` (document index, chunk index, chunks in document). Plus the full documents' word counts and the
reference summaries of validation documents (for ROUGE / BERTScore in P6_02).

In [10]:
def save_tokens(texts, path_stem):
    offs = [0]; n = 0
    with open(str(path_stem) + "_tokens.bin.tmp", "wb") as f:
        for i in range(0, len(texts), 2000):
            for e in TOK.encode_batch(texts[i:i + 2000]):
                a = np.asarray(e.ids, dtype=np.uint16); f.write(a.tobytes()); n += len(a); offs.append(n)
    os.replace(str(path_stem) + "_tokens.bin.tmp", str(path_stem) + "_tokens.bin")
    offs = np.asarray(offs, np.int64); np.save(str(path_stem) + "_offsets.npy", offs); return np.diff(offs)

TOKSTATS = {}
for s, (src, tgt, di, ci, nc, _, _) in PAIRS.items():
    ls = save_tokens(src, P6_DATA / f"{s}_src")
    st = {"chunks": len(src), "src_tok_mean": float(ls.mean()), "src_tok_max": int(ls.max())}
    if DATA[s]["sums"] is not None:
        lt = save_tokens(tgt, P6_DATA / f"{s}_tgt")
        st.update(tgt_tok_mean=float(lt.mean()), tgt_tok_p90=float(np.percentile(lt, 90)), tgt_over_limit=float(np.mean(lt > TGT_TOK)))
    np.savez(P6_DATA / f"{s}_pairs.npz", doc=di, chunk=ci, n_chunks=nc)
    TOKSTATS[s] = st
display(pd.DataFrame(TOKSTATS).T.round(3))
for s in ("train", "val", "test"):
    save_json({"ids": DATA[s]["ids"], "doc_words": [len(x.split()) for x in DATA[s]["docs"]]}, P6_DATA / f"{s}_docs.json")
save_json([{"id": i, "summary": t} for i, t in zip(DATA["val"]["ids"], DATA["val"]["sums"])], P6_DATA / "val_refs.json")
save_json({"ids": DATA["val"]["ids"], "docs": DATA["val"]["docs"]}, P6_DATA / "val_docs_text.json")

,chunks,src_tok_mean,src_tok_max,tgt_tok_mean,tgt_tok_p90,tgt_over_limit
train,62491.0,542.524,606.0,106.323,262.0,0.024
val,3300.0,544.184,601.0,103.872,253.0,0.020
test,1020.0,546.816,600.0,NaN,NaN,NaN


## 8. Baselines on validation (ROUGE as in the Phase-1 evaluator: whitespace tokens, ROUGE-1 / ROUGE-L F1)
* `lead-N`: the first N words of the document, N from the length model.
* `lead-spread`: the first words of every chunk, N in total — covers the whole document.
* `alignment oracle`: the reference sentences re-ordered by chunk (≈ 1.0 if alignment loses nothing).

In [11]:
def lcs_length(a, b):
    if not a or not b: return 0
    if len(a) < len(b): a, b = b, a
    masks = {}
    for j, t in enumerate(a): masks[t] = masks.get(t, 0) | (1 << j)
    state = 0
    for t in b:
        m = masks.get(t, 0); x = m | state; y = (state << 1) | 1; state = x & ~(x - y)
    return bin(state).count("1")
def rouge_l_f1(p, r):
    pt, rt = p.split(), r.split()
    if not pt or not rt: return 0.0
    l = lcs_length(pt, rt)
    return 0.0 if not l else 2 * (l / len(pt)) * (l / len(rt)) / (l / len(pt) + l / len(rt))
def rouge1_f1(p, r):
    pc, rc = Counter(p.split()), Counter(r.split()); ov = sum((pc & rc).values())
    if not ov: return 0.0
    pp, rr = ov / sum(pc.values()), ov / sum(rc.values()); return 2 * pp * rr / (pp + rr)

val = DATA["val"]; srcv, tgtv, div, civ, ncv, _, _ = PAIRS["val"]
def predicted_len(words): return int(np.clip(LEN_MODEL["a"] * max(words, 10) ** LEN_MODEL["b"], 30, 3000))
BASE = {}
cands = {"lead-N": [], "lead-spread": [], "alignment oracle": []}
for i, (doc, ref) in enumerate(zip(val["docs"], val["sums"])):
    N = predicted_len(len(doc.split())); cands["lead-N"].append(" ".join(doc.split()[:N]))
    chs = [srcv[k] for k in np.where(div == i)[0]]; per = max(5, N // max(1, len(chs)))
    cands["lead-spread"].append(" ".join(" ".join(c.split()[:per]) for c in chs))
    cands["alignment oracle"].append(" ".join(tgtv[k] for k in np.where(div == i)[0] if tgtv[k]))
for name, preds in cands.items():
    BASE[name] = {"ROUGE-1": float(np.mean([rouge1_f1(p, r) for p, r in zip(preds, val["sums"])])),
                  "ROUGE-L": float(np.mean([rouge_l_f1(p, r) for p, r in zip(preds, val["sums"])])),
                  "words": float(np.mean([len(p.split()) for p in preds]))}
display(pd.DataFrame(BASE).T.round(4))

,ROUGE-1,ROUGE-L,words
lead-N,0.4601,0.2261,719.5803
lead-spread,0.4818,0.2427,710.2676
alignment oracle,1.0000,0.8402,813.2676


In [12]:
meta = {"tokenizer": TOKENIZER, "src_tok": SRC_TOK, "tgt_tok": TGT_TOK, "len_model": LEN_MODEL,
        "splits": {s: len(DATA[s]["ids"]) for s in ("train", "val", "test")}, "tokens": TOKSTATS, "baselines_val": BASE,
        "summary_words": {"median": float(np.median(sw)), "mean": float(sw.mean())}, "doc_words": {"median": float(np.median(dw)), "mean": float(dw.mean())},
        "summary_source_position_hist": (hist / max(1, hist.sum())).round(4).tolist(), "test_in_train_removed": int(OVERLAP_TEST_IN_TRAIN)}
save_json(meta, P6_DATA / "meta.json")
print(json.dumps({k: v for k, v in meta.items() if k != "tokens"}, indent=1))
print("saved:", sorted(f"{p.name} ({p.stat().st_size/1e6:.1f} MB)" for p in P6_DATA.iterdir()))

{
 "tokenizer": "bpe32768",
 "src_tok": 600,
 "tgt_tok": 384,
 "len_model": {
  "a": 4.6225585455533444,
  "b": 0.6124661450078124
 },
 "splits": {
  "train": 6646,
  "val": 355,
  "test": 100
 },
 "baselines_val": {
  "lead-N": {
   "ROUGE-1": 0.4601006772880081,
   "ROUGE-L": 0.2260664853157791,
   "words": 719.5802816901409
  },
  "lead-spread": {
   "ROUGE-1": 0.4818090452657979,
   "ROUGE-L": 0.24272120177865064,
   "words": 710.2676056338029
  },
  "alignment oracle": {
   "ROUGE-1": 1.0,
   "ROUGE-L": 0.8402071122346804,
   "words": 813.2676056338029
  }
 },
 "summary_words": {
  "median": 639.0,
  "mean": 841.1131266961862
 },
 "doc_words": {
  "median": 3131.0,
  "mean": 4369.46236251964
 },
 "summary_source_position_hist": [
  0.2199,
  0.0664,
  0.0836,
  0.0661,
  0.0562,
  0.122,
  0.0832,
  0.0753,
  0.0992,
  0.1281
 ],
 "test_in_train_removed": 0
}
saved: ['meta.json (0.0 MB)', 'test_docs.json (0.0 MB)', 'test_pairs.npz (0.0 MB)', 'test_src_offsets.npy (0.0 MB)', 'test_